# Entrenar a Tuki en Miskito y Kriol (Gemma 3 1B + LoRA)

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (o mejor).
2. En Hugging Face, acepta la licencia de `google/gemma-3-1b-it` y crea un token de lectura.
3. Ten a mano `gemma_tutor_dataset.zip` (contiene `entrenamiento.jsonl`, `validacion.jsonl`, `prueba.jsonl`).

El resultado final es `gemma3-1b-aikukisna-tutor-int4.task`, que se publica como Release `modelo-tuki-v3-tutor`.
Las métricas automáticas **no sustituyen** la revisión lingüística humana.

In [ ]:
!pip -q install -U "transformers>=4.50" "peft>=0.14" "accelerate>=1.3" datasets sacrebleu sentencepiece
# Colab trae torchao 0.10, que peft rechaza; el entrenamiento no lo necesita. Se desinstala DESPUÉS
# de instalar lo demás, para que ninguna dependencia lo vuelva a traer.
!pip -q uninstall -y torchao
import importlib.util
assert importlib.util.find_spec("torchao") is None, "torchao sigue instalado: reinicia la sesión y repite esta celda"

In [ ]:
from huggingface_hub import login
login()  # pega tu token de Hugging Face

In [ ]:
# Todo se guarda en Google Drive: si Colab corta la sesión (límite de GPU), el entrenamiento se
# retoma desde el último punto de guardado en vez de empezar de cero.
from google.colab import drive, files
import zipfile, os, shutil
drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/aikukisna_gemma"
os.makedirs(CARPETA, exist_ok=True)
ZIP = f"{CARPETA}/gemma_tutor_dataset.zip"
if not os.path.exists(ZIP):
    subido = files.upload()  # selecciona gemma_tutor_dataset.zip (solo la primera vez)
    shutil.copy(next(iter(subido)), ZIP)
zipfile.ZipFile(ZIP).extractall("datos")
print(os.listdir("datos"))

## 1. Cargar modelo y datos

In [ ]:
import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE = "google/gemma-3-1b-it"
tok = AutoTokenizer.from_pretrained(BASE)
# fp16 desborda con Gemma 3 en T4: se entrena en fp32 (1B cabe de sobra en 16 GB con LoRA).
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=dtype, attn_implementation="eager").to("cuda")

def leer(nombre):
    with open(f"datos/{nombre}", encoding="utf8") as f:
        return [json.loads(l) for l in f if l.strip()]

entrenamiento, validacion, prueba = leer("entrenamiento.jsonl"), leer("validacion.jsonl"), leer("prueba.jsonl")
print(len(entrenamiento), len(validacion), len(prueba))

In [ ]:
# Misma plantilla que GemmaTraductorLocal en la app.
def prompt(texto):
    return f"<start_of_turn>user\n{texto}<end_of_turn>\n<start_of_turn>model\n"

MAX_LEN = 160

def codificar(ej):
    p = tok(prompt(ej["prompt"]), add_special_tokens=False)["input_ids"]
    r = tok(ej["respuesta"] + "<end_of_turn>", add_special_tokens=False)["input_ids"]
    ids = ([tok.bos_token_id] + p + r)[:MAX_LEN]
    # Solo se aprende la respuesta, no la instrucción.
    etiquetas = ([-100] * (1 + len(p)) + r)[:MAX_LEN]
    return {"input_ids": ids, "labels": etiquetas}

from datasets import Dataset
ds_entrenamiento = Dataset.from_list(entrenamiento).map(codificar, remove_columns=list(entrenamiento[0].keys()))
ds_validacion = Dataset.from_list(validacion[:400]).map(codificar, remove_columns=list(validacion[0].keys()))

## 2. Medir el modelo base (antes del ajuste)

In [ ]:
import sacrebleu
from collections import defaultdict

@torch.no_grad()
def traducir(textos, lote=16):
    tok.padding_side = "left"
    salidas = []
    for i in range(0, len(textos), lote):
        entrada = tok([tok.bos_token + prompt(t) for t in textos[i:i + lote]], return_tensors="pt",
                      padding=True, add_special_tokens=False).to("cuda")
        gen = modelo.generate(**entrada, max_new_tokens=64, do_sample=False)
        for fila in gen[:, entrada["input_ids"].shape[1]:]:
            salidas.append(tok.decode(fila, skip_special_tokens=True).split("<end_of_turn>")[0].strip())
    return salidas

def evaluar(ejemplos, por_direccion=60):
    grupos = defaultdict(list)
    for e in ejemplos:
        grupos[e["direccion"]].append(e)
    resultado = {}
    for direccion, filas in sorted(grupos.items()):
        filas = filas[:por_direccion]
        hip = traducir([f["prompt"] for f in filas])
        resultado[direccion] = round(sacrebleu.corpus_chrf(hip, [[f["respuesta"] for f in filas]], word_order=2).score, 1)
        print(direccion, resultado[direccion], "| ej:", filas[0]["prompt"].split("\n\n")[1], "→", hip[0])
    return resultado

ARCHIVO_ANTES = f"{CARPETA}/evaluacion_base.json"
if os.path.exists(ARCHIVO_ANTES):
    antes = json.load(open(ARCHIVO_ANTES))  # ya medido en una sesión anterior
else:
    antes = evaluar(prueba)
    json.dump(antes, open(ARCHIVO_ANTES, "w"), indent=2)

## 3. Ajuste fino con LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

modelo.gradient_checkpointing_enable()
modelo.enable_input_require_grads()
modelo = get_peft_model(modelo, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
modelo.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f"{CARPETA}/lora", num_train_epochs=1, per_device_train_batch_size=16,
    # warmup_steps en vez de warmup_ratio: transformers 5 eliminó warmup_ratio (110 ≈ 3 % de ~3 700 pasos).
    gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=110,
    logging_steps=50, eval_strategy="steps", eval_steps=500, save_steps=250, save_total_limit=2,
    bf16=dtype == torch.bfloat16, report_to="none",
)
entrenador = Trainer(
    model=modelo, args=args, train_dataset=ds_entrenamiento, eval_dataset=ds_validacion,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)
import glob
puntos = glob.glob(f"{CARPETA}/lora/checkpoint-*")
if puntos:
    print("Retomando desde", max(puntos, key=lambda r: int(r.rsplit("-", 1)[1])))
entrenador.train(resume_from_checkpoint=bool(puntos))

## 4. Medir después del ajuste

In [ ]:
modelo.eval()
despues = evaluar(prueba)
print("\nchrF++ por dirección (antes → después)")
for d in sorted(despues):
    print(f"{d}: {antes.get(d)} → {despues[d]}")
json.dump({"antes": antes, "despues": despues}, open("evaluacion_prueba.json", "w"), indent=2)
shutil.copy("evaluacion_prueba.json", CARPETA)

## 5. Fusionar LoRA y guardar el modelo completo

In [ ]:
fusionado = modelo.merge_and_unload()
fusionado.save_pretrained("gemma_aikukisna_tutor", safe_serialization=True)
tok.save_pretrained("gemma_aikukisna_tutor")
del modelo, fusionado, entrenador
torch.cuda.empty_cache()

## 6. Convertir al formato del teléfono (.task de MediaPipe)

Necesita bastante RAM: si Colab se queda sin memoria, usa *Entorno con RAM alta*.

In [ ]:
!pip -q install litert-torch mediapipe
# litert-lm-builder (opcional, solo para el formato .litertlm) trae código protobuf 6, pero mediapipe
# deja instalado protobuf 5 y la importación falla con VersionError. litert-torch lo importa dentro de
# un try/except ImportError, así que basta con quitarlo para que convierta a .tflite normalmente.
!pip -q uninstall -y litert-lm-builder litert_lm_builder
import importlib.util, os, shutil
_spec = importlib.util.find_spec("litert_lm_builder")
if _spec and _spec.submodule_search_locations:
    shutil.rmtree(list(_spec.submodule_search_locations)[0])
importlib.invalidate_caches()
assert importlib.util.find_spec("litert_lm_builder") is None, "litert_lm_builder sigue instalado"

In [ ]:
MODELO = "gemma_aikukisna_tutor"  # carpeta con el modelo fusionado (model.safetensors + config.json)

from huggingface_hub import hf_hub_download
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.layers import kv_cache
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig

# save_pretrained solo guarda tokenizer.json; MediaPipe necesita el tokenizer.model de SentencePiece,
# que es el mismo del modelo base (el ajuste no cambia el vocabulario).
TOKENIZADOR = hf_hub_download("google/gemma-3-1b-it", "tokenizer.model", token=globals().get("HF_TOKEN"))

KV_MAX = 1536
pytorch_model = gemma3.build_model_1b(MODELO)
config = ExportConfig()
config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
config.mask_as_input = True
converter.convert_to_tflite(
    pytorch_model,
    output_path="salida_tflite",
    output_name_prefix="gemma3_aikukisna",
    prefill_seq_len=[128, 512],
    kv_cache_max_len=KV_MAX,
    quantize="dynamic_int4_block32",
    export_config=config,
)
!ls -la salida_tflite

In [ ]:
# El mediapipe que se instala en Python 3.13 ya no trae el empaquetador (mediapipe.tasks.python.genai).
# Se crea un Python 3.12 aparte con uv solo para este paso, con una versión que sí lo incluye.
!pip -q install uv
!uv venv -q -p 3.12 /tmp/mp312
!uv pip install -q --python /tmp/mp312 "mediapipe==0.10.18" sentencepiece
import glob, os, subprocess

tflite = glob.glob("salida_tflite/*.tflite")[0]
script = f'''
from mediapipe.tasks.python.genai import bundler
bundler.create_bundle(bundler.BundleConfig(
    tflite_model={tflite!r},
    tokenizer_model={TOKENIZADOR!r},
    start_token="<bos>",
    stop_tokens=["<eos>", "<end_of_turn>"],
    output_filename="gemma3-1b-aikukisna-tutor-int4.task",
))
'''
# Sin prompt_prefix/suffix: GemmaTraductorLocal y GemmaAssistantEngine ya arman los turnos.
# MPLBACKEND=Agg: el cuaderno define un backend de matplotlib que el Python aparte no conoce.
subprocess.run(["/tmp/mp312/bin/python", "-c", script], check=True, env={**os.environ, "MPLBACKEND": "Agg"})
!ls -la gemma3-1b-aikukisna-tutor-int4.task

shutil.copy("gemma3-1b-aikukisna-tutor-int4.task", CARPETA)  # copia de respaldo

## 7. Descargar resultados

Descarga el `.task` y `evaluacion_prueba.json`. El `.task` se publica en GitHub como Release
`modelo-tuki-v3-tutor` (archivo `gemma3-1b-aikukisna-tutor-int4.task`); después, en `local.properties` de la app:

```
TUKI_MODELO_AJUSTADO=true
```

In [ ]:
files.download("evaluacion_prueba.json")
files.download("gemma3-1b-aikukisna-tutor-int4.task")